In [4]:
import os
import shutil
import subprocess
from glob import glob
from tqdm import tqdm  # 진행률 표시 바

In [5]:
def resize_videos_in_place(target_folder, target_width=1280):
    print(f"🎬 [3단계] 동영상 리사이징 (Width {target_width}) 시작: {target_folder}")

    # 처리할 확장자 목록
    extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv']
    video_files = []
    for ext in extensions:
        video_files.extend(glob(os.path.join(target_folder, ext)))

    if not video_files:
        print("   - 동영상 파일이 없습니다.")
        return

    # tqdm으로 진행률 표시
    for file_path in tqdm(video_files, desc="영상 변환 중"):

        # 임시 파일명 생성 (같은 폴더에 생성)
        dir_name = os.path.dirname(file_path)
        file_name = os.path.basename(file_path)
        temp_output = os.path.join(dir_name, f"temp_{file_name}")

        # FFmpeg 명령어
        # scale=1280:-1 -> 가로 1280, 세로는 비율 유지
        # -loglevel error -> 에러만 출력
        command = [
            'ffmpeg', '-i', file_path,
            '-vf', f'scale={target_width}:-1',
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'copy',
            '-y', '-loglevel', 'error',
            temp_output
        ]

        try:
            # 1. 변환 실행
            subprocess.run(command, check=True)

            # 2. 성공 시 원본 삭제 후 임시 파일을 원본 이름으로 변경 (덮어쓰기 효과)
            if os.path.exists(temp_output):
                os.remove(file_path)      # 원본 삭제
                os.rename(temp_output, file_path) # 임시파일 -> 원본명

        except subprocess.CalledProcessError:
            print(f"\n❌ 변환 실패: {file_name}")
            if os.path.exists(temp_output):
                os.remove(temp_output) # 실패한 찌꺼기 파일 삭제

    print("✅ 모든 동영상 리사이징 완료!\n")

In [8]:
target_folder = "/Users/sungminhong/Documents/deepleaning_proj/datasets/asso"
resize_videos_in_place(target_folder, target_width=1280)

🎬 [3단계] 동영상 리사이징 (Width 1280) 시작: /Users/sungminhong/Documents/deepleaning_proj/datasets/asso


영상 변환 중: 100%|██████████| 72/72 [1:03:04<00:00, 52.57s/it]

✅ 모든 동영상 리사이징 완료!

